# 03a - Compute SHAP and LIME attributions for ML models

Computes SHAP (tree explainer) and LIME (tabular) attributions for the
four ML models trained in notebook 02a. Saves attribution matrices as
`.npy` files under `experiments/results/`.


In [ ]:
import numpy as np

from xai_blockchain_framework.config import CONFIG, PATHS, set_seed
from xai_blockchain_framework.models import load_ml_model
from xai_blockchain_framework.utils import save_npy
from xai_blockchain_framework.xai import LimeTabularExplainer, ShapTreeExplainer

set_seed()
N_EVAL = 200  # number of test instances to explain per config


## Load splits and models

In [ ]:
ell = np.load(PATHS.data_processed / 'elliptic_splits.npz')
eth = np.load(PATHS.data_processed / 'ethereum_splits.npz')

# Stratified sampling: ensure a balanced mix of fraud and legitimate
# transactions so that Module 4 has enough ambiguous cases downstream.
# Target: 50% fraud + 50% legit when possible, or all fraud cases if
# fewer than N_EVAL/2 are available.
rng = np.random.default_rng(CONFIG.random_seed)

def stratified_indices(y: np.ndarray, n: int, rng) -> np.ndarray:
    fraud = np.where(y == 1)[0]
    legit = np.where(y == 0)[0]
    n_fraud = min(n // 2, len(fraud))
    n_legit = min(n - n_fraud, len(legit))
    sel_fraud = rng.choice(fraud, size=n_fraud, replace=False)
    sel_legit = rng.choice(legit, size=n_legit, replace=False)
    picked = np.concatenate([sel_fraud, sel_legit])
    rng.shuffle(picked)
    return picked

ell_idx = stratified_indices(ell['y_test'], N_EVAL, rng)
eth_idx = stratified_indices(eth['y_test'], N_EVAL, rng)

print(f'Elliptic: {len(ell_idx)} indices, {int(ell["y_test"][ell_idx].sum())} fraud')
print(f'Ethereum: {len(eth_idx)} indices, {int(eth["y_test"][eth_idx].sum())} fraud')

save_npy(ell_idx, PATHS.results_dir / 'xai_eval_indices_elliptic.npy')
save_npy(eth_idx, PATHS.results_dir / 'xai_eval_indices_ethereum.npy')


## Compute attributions

In [ ]:
configs = [
    ('RandomForest', 'Elliptic', ell['X_train'], ell['X_test'], ell_idx),
    ('LightGBM',     'Elliptic', ell['X_train'], ell['X_test'], ell_idx),
    ('RandomForest', 'Ethereum', eth['X_train'], eth['X_test'], eth_idx),
    ('LightGBM',     'Ethereum', eth['X_train'], eth['X_test'], eth_idx),
]

for model_name, dataset, X_tr, X_te, indices in configs:
    model = load_ml_model(PATHS.models_dir / f'{dataset.lower()}_{model_name.lower()}.joblib')
    ds = dataset.lower(); mn = model_name.lower()

    print(f'  SHAP   {model_name} {dataset}')
    shap_expl = ShapTreeExplainer(model)
    shap_values = shap_expl.explain(X_te, indices)
    save_npy(shap_values, PATHS.results_dir / f'shap_{mn}_{ds}.npy')

    print(f'  LIME   {model_name} {dataset}')
    lime_expl = LimeTabularExplainer(X_tr, random_state=CONFIG.random_seed)
    lime_values = lime_expl.explain(X_te, model.predict_proba, indices)
    save_npy(lime_values, PATHS.results_dir / f'lime_{mn}_{ds}.npy')
